# 04 / Test `SNSNRMetric`

- Author of corrections: Sylvie Dagoret-Campagne
- Creation date: 2026-08-06
- Last update: 2026-08-06
- This notebook is adapted from `03_testSNCadenceMetric.ipynb`.
- It uses `UserPointsSlicer` and a patched `SNSNRMetric` that works with `band`-based v5.3+ OpSim databases.
- The SN reference object used here is a lightweight debug reference. Replace it with a scientific SN reference for production work.

## 1. Overview

- `SNSNRMetric` estimates a detection fraction based on simulated SN SNR curves.
- The metric compares the SNR of the observed cadence to the SNR of fake observations generated from the season properties.
- In recent OpSim databases, the science selection is safer with `band` than with `filter` because `filter` can contain labels such as `r_57`, `g_6`, ...


### 🔬 Ce que ça mesure

Le **signal-to-noise ratio** d’une SN modèle :

- à un redshift donné
- avec un modèle de luminosité

### ⚙️ dépend de :

- `fiveSigmaDepth`
- seeing
- sky brightness
- exposition

### 🧠 Interprétation

👉 “Est-ce que je _vois_ la SN à cette époque ?”

✔️ introduit la physique instrumentale  
❗ mais pas encore la qualité de la courbe de lumière

In [ ]:
import os
import time
import tempfile
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "mplconfig_opsim53"))

%matplotlib inline

import rubin_sim.maf as maf
from rubin_sim.maf.stackers import BaseStacker

try:
    from rubin_sim.data import get_baseline
except ImportError:
    from rubin_scheduler.data import get_baseline

## 2. Configuration

In [ ]:
project_data_dir = Path(os.environ.get("RUBIN_SIM_DATA_DIR", "/Users/dagoret/DATA/OpSim"))
os.environ.setdefault("RUBIN_SIM_DATA_DIR", str(project_data_dir))
print(f"RUBIN_SIM_DATA_DIR = {project_data_dir}")

science_band = "r"

run_db = project_data_dir / "baseline_v5.3.5_10yrs.db"
if not run_db.exists():
    db_candidates = sorted(project_data_dir.glob("*.db"))
    if not db_candidates:
        raise FileNotFoundError(f"No Opsim database found in {project_data_dir}")
    run_db = db_candidates[0]

baseline_file = str(run_db)
print(f"Using Opsim database: {baseline_file}")

In [ ]:
data_dir = None

if data_dir is None:
    data_dir_itself = tempfile.TemporaryDirectory(prefix="04_maf_testSNSNR_", dir=os.getcwd())
    data_dir = data_dir_itself.name

print(f"Using output directory: {data_dir}")

In [ ]:
out_dir = data_dir
resultsDb = maf.db.ResultsDb(out_dir=out_dir)

## 3. View for undertanding SNSNRMetric

In [ ]:
# Inspect the public interface if needed.
%pinfo maf.SNSNRMetric

In [ ]:
# Inspect the source if you want to compare the patched logic below with the installed implementation.
%psource maf.SNSNRMetric

## 4. Build a debug SN reference

In [ ]:
class DemoSNSNRReference:
    """Small synthetic SN reference used to keep the notebook runnable."""

    def __init__(self, names=("demo",)):
        self.names_ref = list(names)
        time_grid = np.linspace(-120.0, 120.0, 600)
        mag_grid = np.linspace(20.0, 28.5, 200)

        self.fluxes = []
        self.mag_to_flux = []

        for _ in self.names_ref:
            # Smooth positive flux curve in arbitrary units.
            flux = 1200.0 * np.exp(-((time_grid / 25.0) ** 2)) + 15.0
            self.fluxes.append(interp1d(time_grid, flux, bounds_error=False, fill_value=(flux[0], flux[-1])))

            # Rough magnitude-to-flux mapping in arbitrary units.
            flux5 = 10.0 ** (-0.4 * (mag_grid - 25.0)) * 100.0
            self.mag_to_flux.append(interp1d(mag_grid, flux5, bounds_error=False, fill_value="extrapolate"))


lim_sn = DemoSNSNRReference(names=("demo",))
print(f"Reference names: {lim_sn.names_ref}")

## 5. Define a season stacker

In [ ]:
class SeasonStacker(BaseStacker):
    cols_added = ["season"]

    def __init__(self, mjd_col: str = "observationStartMJD"):
        self.mjd_col = mjd_col
        self.cols_req = [mjd_col]
        self.units = ["int"]

    def _run(self, sim_data, cols_present=False):
        # Define seasons relative to the first observation in the slice.
        mjd0 = sim_data[self.mjd_col].min()
        season = np.floor((sim_data[self.mjd_col] - mjd0) / 365.25).astype(int)
        sim_data["season"] = season
        return sim_data

## 6. Define the patched metric

In [ ]:
class PatchedSNSNRMetric(maf.SNSNRMetric):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

        # Remove season from the SQL request and request band explicitly.
        cols = [str(c) for c in self.col_name_arr if c != "season"]
        if "band" not in cols:
            cols.append("band")
        self.col_name_arr = np.array(cols, dtype=object)

    def run(self, data_slice, slice_point=None):
        # Keep only standard Rubin bands and use the band column instead of filter.
        good_bands = np.isin(data_slice["band"], self.filter_names)
        data_slice = data_slice[good_bands]
        if data_slice.size == 0:
            return None

        data_slice.sort(order=self.mjd_col)

        if self.season != -1:
            seasons = self.season
        else:
            seasons = np.unique(data_slice["season"])

        if not isinstance(seasons, (list, tuple, np.ndarray)):
            seasons = [seasons]

        self.info_season = None
        for seas in seasons:
            info = self.season_info(data_slice, seas)
            if info is not None and info["season_length"] >= self.shift:
                if self.info_season is None:
                    self.info_season = info
                else:
                    self.info_season = np.concatenate((self.info_season, info))

        self.info_season = self.check_seasons(self.info_season)
        if self.info_season is None:
            return 0.0

        sel = data_slice[np.isin(data_slice["season"], np.array(seasons))]
        detect_frac = None
        if len(sel) >= 5:
            detect_frac = self.process(sel)

        if detect_frac is not None:
            return np.median(detect_frac["frac_obs_{}".format(self.names_ref[0])])
        return 0.0

    def snr_slice(self, data_slice, j=-1, output_q=None):
        # Keep the installed algorithm, but make the returned field names consistent.
        field_ra = np.mean(data_slice[self.ra_col])
        field_dec = np.mean(data_slice[self.dec_col])
        nvisits = np.median(data_slice[self.nexp_col] / 2.0)
        m5 = np.mean(data_slice[self.m5_col])
        exptime = np.median(data_slice[self.exptime_col])
        data_slice.sort(order=self.mjd_col)
        mjds = data_slice[self.mjd_col]
        band = np.unique(data_slice[self.filter_col])[0]

        dates = None
        for val in self.info_season:
            arr = np.arange(val["MJD_min"] + self.shift, val["MJD_max"] + 1.0, 1.0)
            dates = arr if dates is None else np.concatenate((dates, arr))

        t0_lc = dates - self.shift
        time_for_lc = -t0_lc[:, None] + mjds
        phase = time_for_lc / (1.0 + self.z)
        phase_max = self.shift / (1.0 + self.z)
        flag = (phase >= self.min_rf_phase) & (phase <= phase_max)

        m5_vals = np.tile(data_slice[self.m5_col], (len(time_for_lc), 1))
        fluxes_tot, snr = self.snr(time_for_lc, m5_vals, flag, t0_lc)

        _, idx = np.unique(snr["season"], return_inverse=True)
        infos = self.info_season[idx]

        vars_info = ["cadence", "season_length", "MJD_min"]
        snr = np.lib.recfunctions.append_fields(snr, vars_info, [infos[name] for name in vars_info])
        snr = np.lib.recfunctions.append_fields(snr, "DayMax", t0_lc)
        snr = np.lib.recfunctions.append_fields(snr, "MJD", dates)
        snr = np.lib.recfunctions.append_fields(
            snr, "m5_eff", np.mean(np.ma.array(m5_vals, mask=~flag), axis=1)
        )

        global_info = [(field_ra, field_dec, band, m5, nvisits, exptime)] * len(snr)
        names = ["fieldRA", "fieldDec", "band", "m5", "nvisits", "ExposureTime"]
        global_info = np.rec.fromrecords(global_info, names=names)
        snr = np.lib.recfunctions.append_fields(snr, names, [global_info[name] for name in names])

        if output_q is not None:
            output_q.put({j: snr})
        else:
            return snr

    def gen_fakes(self, slice_sel, band):
        # Generate a local synthetic fake-observation table, avoiding the fragile helper API.
        field_ra = np.mean(slice_sel[self.ra_col])
        field_dec = np.mean(slice_sel[self.dec_col])
        fake_obs = []

        for val in self.info_season:
            cadence = val["cadence"]
            mjd_min = val["MJD_min"]
            season_length = val["season_length"]
            nvisits = val["nvisits"]
            m5 = val["m5"]

            mjd = np.arange(mjd_min, mjd_min + season_length + cadence, cadence)
            m5_nocoadd = m5 - 1.25 * np.log10(float(nvisits) * 30.0 / 30.0)

            arr = np.zeros(
                len(mjd),
                dtype=[
                    ("observationStartMJD", "f8"),
                    ("fieldRA", "f8"),
                    ("fieldDec", "f8"),
                    ("filter", "U4"),
                    ("band", "U4"),
                    ("fiveSigmaDepth", "f8"),
                    ("numExposures", "f8"),
                    ("visitExposureTime", "f8"),
                    ("season", "f8"),
                ],
            )
            arr["observationStartMJD"] = mjd
            arr["fieldRA"] = field_ra
            arr["fieldDec"] = field_dec
            arr["filter"] = band
            arr["band"] = band
            arr["fiveSigmaDepth"] = m5_nocoadd
            arr["numExposures"] = nvisits
            arr["visitExposureTime"] = 30.0
            arr["season"] = val["season"]
            fake_obs.append(arr)

        return np.concatenate(fake_obs)

    def snr_fakes(self, data_slice):
        band = np.unique(data_slice[self.filter_col])[0]
        fake_obs = self.gen_fakes(data_slice, band)
        return self.snr_slice(fake_obs[fake_obs["band"] == band])


metric = PatchedSNSNRMetric(lim_sn=lim_sn, names_ref=lim_sn.names_ref, season=-1, coadd=False)
stackers = [SeasonStacker()]

## 7. Quick smoke test

## 8. Configure and run MAF bundles

### 8.1 UserPointsSlicer in COSMOS Field

#### 8.1.1 Define the slicer at one single  point

In [ ]:
RA_COSMOS = 150.1167
DEC_COSMOS = 2.2058

ra_list = [RA_COSMOS]
dec_list = [DEC_COSMOS]

In [ ]:
# Use band-based selection because the database stores values such as u_24, g_6, r_57, ... in the filter column.
sqlconstraint = f'band="{science_band}"'

slicer = maf.slicers.UserPointsSlicer(ra=ra_list, dec=dec_list, use_camera=False, verbose=True)

In [ ]:
# These summaries are the most useful outputs for a single-point slicer.
sn_summary = [maf.metrics.MedianMetric(), maf.metrics.SumMetric(), maf.metrics.MeanMetric()]

### 8.1.2 Create the bundle

In [ ]:
bundle = maf.metric_bundle.MetricBundle(
    metric,
    slicer,
    sqlconstraint,
    stacker_list=stackers,
    summary_metrics=sn_summary,
)

### 8.1.3 Create the group bundle

In [ ]:
bdict = {"sn": bundle}
group = maf.MetricBundleGroup(bdict, baseline_file, out_dir, resultsDb)

### 8.1.4  Run the metric

In [ ]:
start = time.perf_counter()
group.run_all()
elapsed = time.perf_counter() - start
print(f"group.run_all() finished in {elapsed:.1f} s")

bundle.compute_summary_stats()
print("Summary values:")
for name, value in bundle.summary_values.items():
    print(f"  {name}: {value}")

### 8.1.5  Inspect results

In [ ]:
values = np.asarray(bundle.metric_values)
finite = np.isfinite(values)
print("Metric values:", values)
print("Finite count:", finite.sum())
print("Mean:", np.nanmean(values))
print("Median:", np.nanmedian(values))
print("Min:", np.nanmin(values))
print("Max:", np.nanmax(values))

### 8.1.6  Optional plots

In [ ]:
# Plotting is optional here because a UserPointsSlicer is better inspected through the numbers above.
make_plots = True

if make_plots:
    bundle.plot()
    plt.show()

### 8.2 UserPointsSlicer in COSMOS Field with lists of Ra,Dec 

#### 8.2.1 Define the slicer

In [ ]:
ra_list = np.array([150.1167, 150.2, 150.0, 150.1])
dec_list = np.array([2.2058, 2.3, 2.1, 2.25])

slicer = maf.slicers.UserPointsSlicer(ra=ra_list, dec=dec_list, use_camera=False, verbose=True)

bundle = maf.metric_bundle.MetricBundle(
    metric,
    slicer,
    sqlconstraint,
    stacker_list=stackers,
    summary_metrics=sn_summary,
)

bdict = {"sn": bundle}
group = maf.MetricBundleGroup(bdict, baseline_file, out_dir, resultsDb)

#### 8.2.2 Run the metric

In [ ]:
start = time.perf_counter()
group.run_all()
elapsed = time.perf_counter() - start
print(f"group.run_all() finished in {elapsed:.1f} s")

bundle.compute_summary_stats()
print("Summary values:")
for name, value in bundle.summary_values.items():
    print(f"  {name}: {value}")

#### 8.2.3 View summary statistics

In [ ]:
values = np.asarray(bundle.metric_values)
finite = np.isfinite(values)
print("Metric values:", values)
print("Finite count:", finite.sum())
print("Mean:", np.nanmean(values))
print("Median:", np.nanmedian(values))
print("Min:", np.nanmin(values))
print("Max:", np.nanmax(values))

#### 8.2.4 Optional plot

In [ ]:
# Plotting remains optional for the multi-point case as well.
make_plots = True

if make_plots:
    bundle.plot()
    plt.show()

### 8.3 Slicer with HealpixSlicer

#### 8.3.1 Define the slicer

In [ ]:
NSIDE = 16

In [ ]:
slicer = maf.slicers.HealpixSlicer(nside=NSIDE, use_cache=False)

#### 8.3.2 Define the bundle

In [ ]:
bundle = maf.metric_bundle.MetricBundle(
    metric,
    slicer,
    sqlconstraint,
    stacker_list=stackers,
    summary_metrics=sn_summary,
)

#### 8.3.3 Define the bundle group

In [ ]:
bdict = {"sn": bundle}
group = maf.MetricBundleGroup(bdict, baseline_file, out_dir, resultsDb)

#### 8.3.4 Run the metric

In [ ]:
start = time.perf_counter()
group.run_all()
elapsed = time.perf_counter() - start
print(f"group.run_all() finished in {elapsed:.1f} s")

#### 8.3.5 View summary statistics

In [ ]:
bundle.compute_summary_stats()
print("Summary values:")
for name, value in bundle.summary_values.items():
    print(f"  {name}: {value}")

#### 8.3.5 Optional Plots

In [ ]:
# Plotting is optional here because a UserPointsSlicer is better inspected through the numbers above.
make_plots = True

if make_plots:
    bundle.plot()
    plt.show()